In [478]:
import numpy as np
from cuml.kernel_ridge import KernelRidge
from cuml.metrics import pairwise_distances
import tqdm

In [479]:
def loss_fn(x):
	return np.squeeze((x - 1) ** 2, axis=-1)

In [480]:
loss_weighted = True
latent_as_neighbor = True
lambda_ridge = 1.0
k = 8
epsilon = 1e-4
p = 1
n_iters = 200
batch_size = 128
latent_size = 32
parameter_size = 1

In [481]:
model = KernelRidge(alpha=lambda_ridge, kernel="rbf")
rng = np.random.default_rng(123)

In [482]:
latent = rng.standard_normal((batch_size, latent_size))
parameter = rng.standard_normal((batch_size, parameter_size))
model.fit(latent, parameter)

KernelRidge(alpha=1.0, kernel='rbf')

In [483]:
mid_idx = int(k / 2)
for i in tqdm.trange(n_iters):
	latent = rng.standard_normal((batch_size, latent_size))
	parameter = model.predict(latent)
	loss = loss_fn(parameter)
	distances = pairwise_distances(latent if latent_as_neighbor else parameter)
	neighbor_idx = np.argpartition(distances, kth=k - 1)[:, :k]
	best_local_idx = np.argpartition(loss[neighbor_idx], kth=mid_idx)[:, [mid_idx]]
	best_idx = np.take_along_axis(neighbor_idx, best_local_idx)[:, 0]
	best_idx = np.where(loss < loss[best_idx], np.arange(batch_size), best_idx)
	best_parameter = parameter[best_idx]
	
	if loss_weighted:
		best_loss = loss[best_idx]
		reduction = loss - best_loss
		movement = np.linalg.norm(parameter - best_parameter, axis=1)
		sample_weight = ((reduction + epsilon) / (movement + epsilon)) ** p
		model.fit(latent, best_parameter, sample_weight)
	else:
		model.fit(latent, best_parameter)

100%|██████████| 200/200 [00:02<00:00, 73.36it/s]


## Results

In [484]:
print(parameter)

[[0.59170709]
 [0.7267449 ]
 [0.53840086]
 [0.74604369]
 [0.85687751]
 [0.46885521]
 [0.38439366]
 [0.85438001]
 [0.92802554]
 [0.65231046]
 [0.71880742]
 [0.69832917]
 [0.6723517 ]
 [0.70356426]
 [0.77998275]
 [0.78203809]
 [0.70964188]
 [0.70944318]
 [0.78233575]
 [0.80458111]
 [0.8801753 ]
 [0.52767046]
 [0.5061284 ]
 [0.70480961]
 [0.69218497]
 [0.7519839 ]
 [0.72632674]
 [0.90372672]
 [0.76332371]
 [0.69895004]
 [0.71528598]
 [0.49976223]
 [0.81588454]
 [1.07285183]
 [0.88315584]
 [0.88165788]
 [0.63646386]
 [0.64034913]
 [0.75475933]
 [0.37208973]
 [0.84859417]
 [0.56289719]
 [0.65437543]
 [0.64887803]
 [0.6148154 ]
 [0.4766922 ]
 [0.80259968]
 [0.50933976]
 [0.87586992]
 [0.65440855]
 [0.70155912]
 [0.73210938]
 [0.61038869]
 [0.785352  ]
 [0.64861105]
 [0.50121602]
 [0.78418013]
 [0.54209622]
 [0.58818801]
 [0.72745867]
 [0.75113456]
 [0.73227693]
 [0.71447517]
 [0.83491717]
 [1.02265631]
 [0.78786779]
 [0.52317894]
 [0.75593303]
 [0.47957246]
 [0.704881  ]
 [0.82518783]
 [0.79

In [475]:
print(loss_fn(parameter))

[1.07636729e-02 1.37901305e-01 1.33689770e-03 1.31776909e-01
 9.06206831e-02 2.87122628e-01 1.13455873e-01 1.36916577e-01
 6.85193516e-02 3.59633944e-02 2.11196982e-03 8.97039654e-04
 2.99125042e-02 9.85941949e-03 1.29261383e-01 3.15503693e-02
 5.82985615e-02 6.09316462e-03 1.54050578e-01 2.16648158e-02
 4.20743297e-02 9.31546099e-02 8.20510398e-02 6.54031697e-02
 2.67298515e-02 3.44418511e-02 2.17288747e-06 1.28203628e-01
 6.73797832e-02 2.19780431e-01 6.36690702e-02 3.74931116e-02
 1.79729769e-02 6.56231611e-02 4.70487122e-02 1.28384344e-01
 4.90232844e-04 1.37417710e-03 9.93259005e-04 5.22117538e-02
 2.46933260e-01 2.96849499e-02 5.25990048e-03 9.53950007e-02
 4.77358622e-02 5.02971680e-02 4.50255463e-03 1.41086202e-01
 4.86115026e-02 1.23833798e-02 1.52438296e-01 2.89374303e-01
 7.98893727e-02 1.10576824e-01 1.78865556e-01 1.34129741e-02
 4.86802878e-02 8.22058512e-02 7.92907220e-02 5.24736360e-02
 4.55880798e-02 8.71233176e-02 2.72702388e-02 7.78165446e-02
 5.97215264e-02 7.216796